Typology classification for LULC transitions

In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np

In [ ]:
# Rule Functions for Crop-Sequence Pattern Detection
def is_monoculture(sequence, target_crop, threshold=0.85):
    clean_seq = [g for g in sequence if g and g != 'nan']
    freq = clean_seq.count(target_crop) / len(clean_seq) if len(clean_seq) > 0 else 0
    return freq >= threshold

In [ ]:
def is_staple_crop_rotation(sequence):
    staple_crops = {'Corn', 'Wheat', 'Soybeans'}
    clean_seq = [g for g in sequence if g not in {'nan', '', None}]
    staple_count = sum(1 for g in clean_seq if g in staple_crops)
    return (staple_count / len(clean_seq)) >= 0.75 and len(set(clean_seq)) > 1

In [ ]:
def is_staple_crop_conversion(sequence):
    staple_crops = {'Corn', 'Wheat', 'Soybeans'}
    non_staple_cropland = {
        'Grains', 'Oilseeds', 'Legumes and Pulses', 'Forage and Cover Crops', 
        'Vegetables', 'Fruits and Nuts', 'Other Crops', 'Specialty Crops',
        'Sugar and Fiber Crops', 'Double Crops', 'Fallowing', 
        'Urban Area', 'Full Solar', 'Partial Solar', 'Forest', 
        'Pasture/Hay', 'Shrubland','Waterbody', 'Barren Land', 'Other Land'
    }

    start_seq = sequence[:11]
    end_seq = sequence[-5:]

    start_staple_ratio = sum(1 for g in start_seq if g in staple_crops) / len(start_seq)
    start_non_staple_ratio = sum(1 for g in start_seq if g in non_staple_cropland) / len(start_seq)
    end_staple_ratio = sum(1 for g in end_seq if g in staple_crops) / len(end_seq)

    return start_staple_ratio <= 0.3636 and start_non_staple_ratio >= 0.6364 and end_staple_ratio >= 0.6


In [ ]:
def is_other_cropland_rotation(sequence):
    staple_crops = {'Corn', 'Wheat', 'Soybeans'}
    other_crops = {
        'Grains', 'Oilseeds', 'Legumes and Pulses', 'Forage and Cover Crops',
        'Vegetables', 'Fruits and Nuts', 'Other Crops', 'Specialty Crops',
        'Sugar and Fiber Crops', 'Double Crops', 'Fallowing'
    }
    
    clean_seq = [g for g in sequence if g not in {'nan', '', None}]
    if not clean_seq:
        return False

    other_ratio = sum(1 for g in clean_seq if g in other_crops) / len(clean_seq)
    has_staple = any(g in staple_crops for g in clean_seq)
    has_other_land = any(g not in (staple_crops | other_crops) for g in clean_seq)

    return other_ratio >= 0.75 and (has_staple or has_other_land)

In [ ]:
def is_non_staple_crop_conversion(sequence):
    staple_and_other_land = {
        'Corn', 'Wheat', 'Soybeans', 'Urban Area', 'Full Solar', 
        'Partial Solar', 'Forest', 'Pasture/Hay', 'Shrubland', 
        'Waterbody', 'Barren Land', 'Other Land'
    }
    non_staple_cropland = {
        'Grains', 'Oilseeds', 'Legumes and Pulses', 'Forage and Cover Crops', 
        'Vegetables', 'Fruits and Nuts', 'Other Crops', 'Specialty Crops',
        'Sugar and Fiber Crops', 'Double Crops', 'Fallowing'
    }

    start_seq = sequence[:11]
    end_seq = sequence[-5:]

    start_staple_other_ratio = sum(1 for g in start_seq if g in staple_and_other_land) / len(start_seq)
    end_staple_other_ratio = sum(1 for g in end_seq if g in staple_and_other_land) / len(end_seq)
    start_non_staple_ratio = sum(1 for g in start_seq if g in non_staple_cropland) / len(start_seq)

    return start_staple_other_ratio <= 0.3636 and start_non_staple_ratio >= 0.6364 and end_staple_other_ratio >= 0.6

In [ ]:
def is_cropland_intensification(sequence):
    non_cropland_set = {
        'Urban Area', 'Full Solar', 'Partial Solar', 'Forest', 
        'Pasture/Hay', 'Shrubland', 'Waterbody', 'Barren Land', 'Other Land'
    }
    cropland_set = {
        'Grains', 'Oilseeds', 'Legumes and Pulses', 'Forage and Cover Crops', 
        'Vegetables', 'Fruits and Nuts', 'Other Crops', 'Specialty Crops',
        'Sugar and Fiber Crops', 'Double Crops', 'Fallowing',
        'Corn', 'Soybeans', 'Wheat'
    }

    start_seq = sequence[:11]
    end_seq = sequence[-5:]

    start_non_cropland_ratio = sum(1 for g in start_seq if g in non_cropland_set) / len(start_seq)
    start_cropland_ratio = sum(1 for g in start_seq if g in cropland_set) / len(start_seq)
    end_cropland_ratio = sum(1 for g in end_seq if g in cropland_set) / len(end_seq)

    return start_non_cropland_ratio >= 0.6364 and start_cropland_ratio <= 0.3636 and end_cropland_ratio >= 0.6


In [ ]:
def is_cropland_reduction(sequence):
    cropland_set = {
        'Corn', 'Wheat', 'Soybeans', 'Grains', 'Oilseeds', 'Legumes and Pulses',
        'Forage and Cover Crops', 'Vegetables', 'Fruits and Nuts', 'Other Crops', 
        'Specialty Crops', 'Sugar and Fiber Crops', 'Double Crops', 'Fallowing'
    }
    non_cropland_set = {
        'Urban Area', 'Full Solar', 'Partial Solar', 'Forest', 'Pasture/Hay', 
        'Shrubland', 'Waterbody', 'Barren Land', 'Other Land'
    }

    start_seq = sequence[:11]
    end_seq = sequence[-5:]

    start_cropland_ratio = sum(1 for g in start_seq if g in cropland_set) / len(start_seq)
    start_non_cropland_ratio = sum(1 for g in start_seq if g in non_cropland_set) / len(start_seq)
    end_non_cropland_ratio = sum(1 for g in end_seq if g in non_cropland_set) / len(end_seq)

    return start_cropland_ratio >= 0.6364 and start_non_cropland_ratio <= 0.3636 and end_non_cropland_ratio >= 0.6


In [ ]:
def is_pasture_conversion(sequence):
    pasture_set = {'Pasture/Hay'}
    non_pasture_set = {
        'Urban Area', 'Full Solar', 'Partial Solar', 'Forest', 'Shrubland',
        'Waterbody', 'Barren Land', 'Other Land', 'Grains', 'Oilseeds',
        'Legumes and Pulses', 'Forage and Cover Crops', 'Vegetables', 'Fruits and Nuts',
        'Other Crops', 'Specialty Crops', 'Sugar and Fiber Crops', 
        'Double Crops', 'Fallowing', 'Corn', 'Soybeans', 'Wheat'
    }

    start_seq = sequence[:11]
    end_seq = sequence[-5:]

    start_non_pasture_ratio = sum(1 for g in start_seq if g in non_pasture_set) / len(start_seq)
    start_pasture_ratio = sum(1 for g in start_seq if g in pasture_set) / len(start_seq)
    end_pasture_ratio = sum(1 for g in end_seq if g in pasture_set) / len(end_seq)

    return start_non_pasture_ratio >= 0.6364 and start_pasture_ratio <= 0.3636 and end_pasture_ratio >= 0.6


In [ ]:
def is_pasture_persistence(sequence):
    pasture_set = {'Pasture/Hay'}
    return (sum(1 for g in sequence if g in pasture_set) / len(sequence)) >= 0.85

In [ ]:
def is_urban_persistence(sequence):
    return (sequence.count('Urban Area') / len(sequence)) >= 0.85

In [ ]:
def is_urban_conversion(sequence):
    urban_set = {'Urban Area'}
    non_urban_set = {
        'Full Solar', 'Partial Solar', 'Forest', 'Shrubland', 'Waterbody',
        'Barren Land', 'Other Land', 'Grains', 'Oilseeds', 'Legumes and Pulses',
        'Forage and Cover Crops', 'Vegetables', 'Fruits and Nuts', 'Other Crops',
        'Specialty Crops', 'Sugar and Fiber Crops', 'Double Crops', 'Fallowing',
        'Corn', 'Soybeans', 'Wheat', 'Pasture/Hay'
    }

    start_seq = sequence[:11]
    end_seq = sequence[-5:]

    start_non_urban_ratio = sum(1 for g in start_seq if g in non_urban_set) / len(start_seq)
    start_urban_ratio = sum(1 for g in start_seq if g in urban_set) / len(start_seq)
    end_urban_ratio = sum(1 for g in end_seq if g in urban_set) / len(end_seq)

    return start_non_urban_ratio >= 0.6364 and start_urban_ratio <= 0.3636 and end_urban_ratio >= 0.6

In [ ]:
def is_full_solar_conversion(sequence):
    return sequence[-1] == 'Full Solar'

def is_partial_solar_conversion(sequence):
    return sequence[-1] == 'Partial Solar'

In [ ]:
def is_natural_land_persistence(sequence):
    natural_set = {'Forest', 'Shrubland', 'Waterbody', 'Barren Land'}
    return (sum(1 for g in sequence if g in natural_set) / len(sequence)) >= 0.85

In [ ]:
# Label Assignment Function
def assign_subtypology_label(sequence):
    sequence = [str(g).strip() for g in sequence if pd.notna(g) and str(g).strip() != '']
    if not sequence:
        return 'Other Mixed Pattern (MxOt)'

    if is_monoculture(sequence, 'Corn'):
        return 'Corn Monoculture (CnMo)'
    elif is_monoculture(sequence, 'Wheat'):
        return 'Wheat Monoculture (WtMo)'
    elif is_monoculture(sequence, 'Soybeans'):
        return 'Soybean Monoculture (SbMo)'
    elif is_staple_crop_rotation(sequence):
        return 'Staple Crop Rotation (StRot)'
    elif is_other_cropland_rotation(sequence):
        return 'Other Cropland Rotation (OcRot)'
    elif is_urban_persistence(sequence):
        return 'Urban Persistence (UrPe)'
    elif is_cropland_intensification(sequence):
        return 'Cropland Intensification (CrIn)'
    elif is_full_solar_conversion(sequence):
        return 'Solar Conversion (SoCv)'
    elif is_partial_solar_conversion(sequence):
        return 'Partial Solar Conversion (SoPr)'
    elif is_cropland_reduction(sequence):
        return 'Cropland Reduction (CrRd)'
    elif is_natural_land_persistence(sequence):
        return 'Natural Land Persistence (Nat)'
    elif is_staple_crop_conversion(sequence):
        return 'Staple Crop Conversion (StCv)'
    elif is_non_staple_crop_conversion(sequence):
        return 'Other Cropland Conversion (OcCv)'
    elif is_pasture_conversion(sequence):
        return 'Pasture/Hay Conversion (PaCv)'
    elif is_pasture_persistence(sequence):
        return 'Pasture/Hay Persistence (PaPe)'
    elif is_urban_conversion(sequence):
        return 'Urban Area Conversion (UrCv)'
    else:
        return 'Other Mixed Pattern (MxOt)'

In [ ]:
# Load the combined farm parcel dataset
data_path = r"E:\Typology_Work\Typology_2\FP_Crop_Names_and_Categories\FP_Primary_HCC_2008_2023.csv"
crop_df = pd.read_csv(data_path)
crop_df.head().columns

In [ ]:
# Extract columns corresponding to yearly crop groups
yearly_crop_cols = sorted([col for col in crop_df.columns if col.startswith('Updated_Group_')])

# Fill missing values with placeholder "NA" to maintain sequence length
crop_df[yearly_crop_cols] = crop_df[yearly_crop_cols].fillna("NA")

In [ ]:
# Create a single sequence string per parcel for pattern detection
crop_df['crop_group_sequence'] = crop_df[yearly_crop_cols].agg('-'.join, axis=1)

# inspect the first few sequences
print(crop_df[['crop_group_sequence']].head())

In [9]:
# Extract unique crop group sequences
unique_sequences = crop_df['crop_group_sequence'].unique()

# Assign conversion/sub-typology labels to each unique sequence
labeled_sequences_df = pd.DataFrame({
    'crop_group_sequence': unique_sequences,
    'conversion_label': [assign_subtypology_label(seq.split('-')) for seq in unique_sequences]
})

# Sort sequences by their conversion label
labeled_sequences_df = labeled_sequences_df.sort_values(by='conversion_label')

In [ ]:
# Extract and print all unique labels
unique_labels = labeled_sequences_df['conversion_label'].unique()
print("Unique Conversion Labels:", unique_labels)

# Count the total number of unique labels
print("\nTotal Number of Unique Conversion Labels:", len(unique_labels))

In [ ]:
#Display counts for each conversion label
conversion_label_counts = labeled_sequences_df['conversion_label'].value_counts()
print("Counts per conversion label:", conversion_label_counts)

In [ ]:
#Apply conversion labels to the full dataset
crop_df['full_conversion_label'] = crop_df['crop_group_sequence'].apply(
    lambda seq: assign_subtypology_label(seq.split('-'))
)

#Split into descriptive name and acronym
crop_df['Conversion_Class'] = crop_df['full_conversion_label'].str.extract(r'^(.*?)\s*\(')[0].str.strip()
crop_df['Conversion_Label'] = crop_df['full_conversion_label'].str.extract(r'\((.*?)\)')[0]

In [ ]:
# Preview labeled dataset
print("\n Sample labeled data:")
print(crop_df[['CSBID', 'Conversion_Class', 'Conversion_Label']].head())

In [ ]:
# Frequency Summary
conversion_counts_df = (
    crop_df['Conversion_Class']
    .value_counts()
    .rename_axis('Conversion_Class')   #name the index
    .reset_index(name = 'Count')       #name the counts column
)

print("\n Conversion Class Counts:\n")
for _, row in conversion_counts_df.iterrows():
    print(f"{row['Conversion_Class']:<35} {row['Count']:>8,}")

# Save Labeled Dataset (Optional)
output_file = r"E:\Typology_Work\Typology_2\results\lulc_transition_typology.csv"
crop_df.to_csv(output_file, index=False)
print(f"\n Labeled dataset saved to: {output_file}")


Land Management Practice Trend Typology

In [ ]:
# Define a function to classify each row based on yearly patterns
def classify_practice_subtypology(parcel_row, year_list):

    irrigation_pattern = []
    conservation_pattern = []
    tile_drain_applied = parcel_row.get('BF_Mean_TileDrain') == 'TD'

    for year in year_list:
            ir_col = f'BF_Mean_Irri_{year}'
            ce_col = f'CE_Status_{year}'

            irrigation_pattern.append(1 if ir_col in parcel_row and parcel_row[ir_col] == 'IR' else 0)
            conservation_pattern.append(1 if ce_col in parcel_row and parcel_row[ce_col] == 'CE' else 0)

    irrigation_ratio = sum(irrigation_pattern) / len(irrigation_pattern)
    conservation_ratio = sum(conservation_pattern) / len(conservation_pattern)

    return classify_practice(
        irrigation_ratio,
        conservation_ratio,
        tile_drain_applied,
        irrigation_pattern,
        year_list
    )

In [ ]:
#Define the final classification logic
def classify_practice(ir_ratio, ce_ratio, td_applied, ir_pattern, year_list):

    years_as_int = [str(y) for y in range(2008, 2024)]
    irrigated_years = np.array(years_as_int)[np.array(ir_pattern) == 1].astype(int)
    irrigation_gaps = np.diff(irrigated_years) if len(irrigated_years) > 1 else []
    avg_gap = np.mean(irrigation_gaps) if len(irrigation_gaps) > 0 else None

    # Irrigation Persistence
    if ir_ratio >= 0.90 and ce_ratio == 0 and not td_applied:
        return 'Irrigation Persistence'
    elif ir_ratio >= 0.90 and ce_ratio > 0 and not td_applied:
        return 'Irrigated and Conserved'
    elif ir_ratio >= 0.90 and td_applied and ce_ratio == 0:
        return 'Irrigated and Tile Drained'
    elif ir_ratio >= 0.90 and ce_ratio > 0 and td_applied:
        return 'Irrigated, Conserved and Tile Drained'

    # Rainfed but Conserved & Tiled 
    elif ir_ratio == 0 and ce_ratio > 0 and td_applied:
        return 'Rainfed, Conserved and Tile Drained'

    # Intermittent Irrigation 
    elif avg_gap is not None and 1.5 <= avg_gap <= 3.5 and ce_ratio == 0 and not td_applied:
        return 'Intermittently Irrigated'
    elif avg_gap is not None and 1.5 <= avg_gap <= 3.5 and ce_ratio > 0 and not td_applied:
        return 'Intermittently Irrigated and Conserved'
    elif avg_gap is not None and 1.5 <= avg_gap <= 3.5 and ce_ratio == 0 and td_applied:
        return 'Intermittently Irrigated and Tile Drained'
    elif avg_gap is not None and 1.5 <= avg_gap <= 3.5 and ce_ratio > 0 and td_applied:
        return 'Intermittently Irrigated, Conserved and Tile Drained'

    # Conservation Transition 
    elif ce_ratio > 0 and not td_applied and ir_ratio == 0:
        return 'Conservation Transition'

    # Tile Drained only 
    elif ir_ratio == 0 and ce_ratio == 0 and td_applied:
        return 'Tile Drained'

    # Default 
    else:
        return 'Conventional'


In [ ]:
# Load Input File
input_csv_path = (r"E:\Typology_Work\Typology_2\Zonal Outputs Land Practices and Farm Parcels\FP_Typology_Final_1\FP_GLR_0823_Conversion_Label.csv")

farm_df = pd.read_csv(input_csv_path)
# farm_df.head().columns

# Define year range (2008–2023)
year_range = list(range(2008, 2024))

In [ ]:
#Apply classification
crop_df['Practice_Class'] = farm_df.apply(
    lambda row: classify_practice_subtypology(row, year_range),
    axis=1
)

# Map full class names to short practice labels
practice_label_map = {
    'Irrigation Persistence': 'IR',
    'Irrigated and Conserved': 'IRCT',
    'Irrigated and Tile Drained': 'IRTD',
    'Irrigated, Conserved and Tile Drained': 'IRCTTD',
    'Rainfed, Conserved and Tile Drained': 'RCTTD',
    'Intermittently Irrigated': 'IIR',
    'Intermittently Irrigated and Conserved': 'IIRCT',
    'Intermittently Irrigated and Tile Drained': 'IIRTD',
    'Intermittently Irrigated, Conserved and Tile Drained': 'IIRCTTD',
    'Conservation Transition': 'CT',
    'Tile Drained': 'TD',
    'Conventional': 'Con'
}

crop_df['Practice_Label'] = crop_df['Practice_Class'].map(practice_label_map)

In [ ]:
#print summary
print("\n Practice Class Counts:\n")
for class_name, count in crop_df['Practice_Class'].value_counts().sort_index().items():
    print(f"{class_name:<45} {count:>10,}")

In [ ]:
#Save output
output_csv_path = r"E:\Typology_Work\Typology_2\results\lulc_transition_lmp_trend_typology.csv"
crop_df.to_csv(output_csv_path, index=False)

print(f"File updated and saved in: {output_csv_path}")

Merge the lulc transition and the lmp trend typology to get the integrated typology

In [ ]:
# Create Integrated Typology Fields
crop_df = pd.read_csv(r"E:\Typology_Work\Typology_2\results\lulc_transition_lmp_trend_typology.csv")

crop_df['Typology_Class'] = (crop_df['Conversion_Class'].astype(str) + ' - ' + crop_df['Practice_Class'].astype(str))

crop_df['Typology_Label'] = (crop_df['Conversion_Label'].astype(str) + '-' + crop_df['Practice_Label'].astype(str))

crop_df['Typology_Descriptor'] = (crop_df['Conversion_Label'].astype(str) + ' - ' + crop_df['Practice_Class'].astype(str))

In [ ]:
# Frequency Table (Full Class Names)
print("\n Counts by Typology_Class (Full Descriptive Names):\n")

class_counts = (
    crop_df['Typology_Class'].value_counts().sort_index())

# Clean formatted printing
max_label_len = class_counts.index.str.len().max()

for label, count in class_counts.items():
    print(f"{label:<{max_label_len}}   {count:>10,}")

In [ ]:
#Save output

integrated_path = r"E:\Typology_Work\Typology_2\results\integrated_typology.csv"
crop_df.to_csv(integrated_path, index = False)

print("Integrated typology saved successfully.")

Integrated typology saved successfully.


In [ ]:
crop_df.head()

In [ ]:
unique_typo=crop_df['Typology_Label'].unique()

print("Unique typology counts:", len(unique_typo))